# Tutorial: 牛津 Tutorial LLM 仿真 - 商业模式画布 + 投资评估 (v6.0 学习科学层)

## Persona Prompt (Oxford Tutorial Fellow)

You are an Oxford tutorial fellow in 商业模式画布与投资评估 (Business Model Canvas + Investment Valuation).

**核心约束 (Persona Constraints):**
- **不直接给答案 (Never give direct answers - Socratic only)**: 每轮以追问结尾, 禁止给出 NPV/IRR/P(NPV>0) 的最终数值或代码答案
- **Socratic questioning (苏格拉底追问)**: 每轮结束必须以一个 probing question 结尾, 迫使学生自己推理
- **Reject vague claims (拒绝含糊断言)**: '差一点' / '差不多' / '感觉可行' 必须用数字辩护
- **Devil's advocate (魔鬼代言人)**: 当学生说 'NPV是正的所以投资', 反驳 'P(NPV>0)只有55.7%, 你愿意拿自己的钱赌55%吗? 反例: 若推理成本上升1pct呢?'
- **Multi-turn scaffolding fade (多轮脚手架渐退)**: 每轮 defense 失败则降一级 scaffold_level (3->2->1->0), 仍禁直接答案
- **限频 (Usage Limit)**: 每单元每天 1 次 tutorial (1次/天), 防止学生把 tutor 当答案机

**话题域 (Topic Domain):**
- 商业模式画布九宫格 AI 适配 (推理成本 + 数据成本 + Agent 渠道 + outcome-based 收入流)
- numpy-financial NPV/IRR/PI/回收期 (MarketingAgent Pro 基准: NPV=$451.2K, IRR=20.08%, PI=1.45)
- scipy.stats 蒙特卡洛 10000 次 + P(NPV>0)=55.7%
- 天道推演 Bull/Base/Bear 三路径 × 三层 (immediate/near/far)
- 推理成本对 AI 估值影响 (DeepSeek 效应, 龙卷风图高杠杆因子)

> 本 tutorial 仿真 Oxford 1对1-3 每周强制口头辩护传统 (Vygotsky 共构 + Socratic 法), 用静态 if/else 模拟 LLM 响应, 不真调 API。


## Pre-Tutorial Task (强制 retrieval - 学生先提交, 测试效应)

Tutorial 前学生必须独立完成并提交以下三项 (强制提取练习, Butler 2010 检索练习证据: 推断题 68% vs 重学 44%):

1. **画布构建 (ILO1)**: 用 pandas 构建一个 AI 产品(任选, 如 CodingAgent)的九宫格 DataFrame, 标注 AI 适配的 4 项独有变化 (推理成本 + 数据成本 + Agent渠道 + outcome-based收入流)
2. **DCF 计算 (ILO2)**: 给定 5 年现金流 `[-100, 30, 40, 50, 60, 70]` (单位 $K), 用 numpy-financial 计算 NPV (10% 折现率) 和 IRR, 并解释 cashflows[0] 为何必为负
3. **决策辩护 (ILO3)**: MarketingAgent Pro 的 NPV=$451.2K, IRR=20.08%, P(NPV>0)=55.7% - 你投资吗?为什么? 用天道推演 Bull/Base/Bear 三路径各推演 immediate 一层

**未提交全部三项不允许进入 tutorial** (强制 retrieval, 防止学生空手来听讲解)。


In [ ]:
# Socratic Multi-Turn Loop (静态 if/else 模拟 LLM 响应, 不调 API)
# 每轮检测 defense 失败则降一级 scaffold, 仍禁直接答案
# >=4 轮 Socratic 追问, >=5 个 probing questions

import json

def socratic_loop(student_submission):
    """4-5 轮 Socratic 追问, 每轮静态分支模拟 LLM 响应, 禁直接答案.
    
    student_submission: dict, 学生 pre-task 提交的内容
    返回: transcript (list of turn dicts)
    """
    transcript = []
    scaffold_level = 3  # 3=高脚手架, 2=中, 1=低, 0=独立辩护
    
    # ===== 轮 1: 画布 AI 独有项 (ILO1) =====
    q1 = ("你的画布里'成本结构'格填了什么? "
          "为什么传统 SaaS 画布没有'推理成本'而 AI 画布必须有? "
          "凭什么说推理成本是 AI 独有? 反例: HubSpot 78% 毛利率, AI SaaS 为何拉低到 65%?")
    transcript.append({"turn": 1, "scaffold": scaffold_level, "question": q1})
    if "推理成本" not in student_submission.get("canvas_cost", ""):
        scaffold_level -= 1  # 降级
        transcript.append({
            "turn": 1,
            "feedback": "漏了推理成本. 反例: HubSpot 78%毛利率, AI SaaS因推理成本拉低到65%. 这意味着你的毛利率假设错了. 重画成本结构格.",
            "scaffold_dropped_to": scaffold_level
        })
    
    # ===== 轮 2: DCF 符号 (ILO2) =====
    q2 = ("你的 NPV 计算中, cashflows[0] 为什么是负数? "
          "如果写成正数会怎样? "
          "npf.irr() 要求至少一次符号变化, 为什么? "
          "假设现金流全正, IRR 会返回什么?")
    transcript.append({"turn": 2, "scaffold": scaffold_level, "question": q2})
    if student_submission.get("npv_sign", "") != "负":
        scaffold_level -= 1
        transcript.append({
            "turn": 2,
            "feedback": "符号错了. NPV公式中 t=0 是投资支出, 必须为负. npf.irr() 要求至少一次符号变化否则无解. 重写 cashflows 序列.",
            "scaffold_dropped_to": scaffold_level
        })
    
    # ===== 轮 3: 蒙特卡洛 vs 点估计 (ILO3) =====
    q3 = ("你说 NPV=$451.2K, 但蒙特卡洛 P(NPV>0)=55.7%. "
          "这两个数矛盾吗? 若不矛盾, 依据是什么? "
          "如何向 CFO 解释为何投 55% 胜率的赌注? "
          "若 P(NPV>0) 降到 40% 呢?")
    transcript.append({"turn": 3, "scaffold": scaffold_level, "question": q3})
    if "分布" not in student_submission.get("mc_explain", ""):
        scaffold_level -= 1
        transcript.append({
            "turn": 3,
            "feedback": "没说'分布'. NPV $451.2K 是均值, P(NPV>0)=55.7% 是分布右偏程度. 两者不矛盾, 是同分布的不同统计量. 蒙特卡洛评估参数不确定性.",
            "scaffold_dropped_to": scaffold_level
        })
    
    # ===== 轮 4: 天道推演三层 (ILO3) =====
    q4 = ("你的 Bull 路径 immediate/near/far 三层各推演了什么? "
          "假设推理成本因 DeepSeek 效应降 90%, 你的 Bull 路径 far 层会怎么变? "
          "反例: 若竞品先降价呢? "
          "天道推演与蒙特卡洛评估的不确定性有什么不同?")
    transcript.append({"turn": 4, "scaffold": scaffold_level, "question": q4})
    if "三层" not in student_submission.get("tiandao", ""):
        scaffold_level = max(0, scaffold_level - 1)
        transcript.append({
            "turn": 4,
            "feedback": "天道推演必须三层(immediate/near/far). 蒙特卡洛评估参数不确定性, 天道推演评估场景路径优不优, 两者互补. 重做 Bull 路径三层.",
            "scaffold_dropped_to": scaffold_level
        })
    
    # ===== 轮 5: 综合反思 (Feed-Forward) =====
    q5 = ("综合以上四轮, 你认为 MarketingAgent Pro 最大的估值风险是什么? "
          "如何用天道推演的 Bull/Base/Bear 重新评估? "
          "若只能改一个假设, 你改哪个? 为什么? "
          "依据是什么?")
    transcript.append({"turn": 5, "scaffold": max(scaffold_level, 0), "question": q5})
    
    return transcript

# ===== 模拟学生提交 (实际从 student_model.json 读) =====
demo_submission = {
    "canvas_cost": "数据成本",   # 漏了推理成本, 触发轮1降级
    "npv_sign": "负",            # 正确
    "mc_explain": "均值和分布",   # 包含'分布'
    "tiandao": "两层推演"        # 漏了'三层', 触发轮4降级
}

result = socratic_loop(demo_submission)
print(json.dumps(result, ensure_ascii=False, indent=2))


In [ ]:
# student_model.json 读写 (记录掌握度/盲点, 跨单元复用)
# Vygotsky 共构: tutor 根据学生模型调整 scaffold_level

import json
import os
from datetime import datetime

STUDENT_MODEL_PATH = "student_model.json"

def load_student_model():
    """加载学生模型, 不存在则初始化"""
    if os.path.exists(STUDENT_MODEL_PATH):
        with open(STUDENT_MODEL_PATH, encoding="utf-8") as f:
            return json.load(f)
    return {
        "unit": "skill4-day5-business-model-canvas-investment",
        "mastery": {
            "ILO1_canvas": 0.0,
            "ILO2_dcf": 0.0,
            "ILO3_monte_carlo_tiandao": 0.0
        },
        "blind_spots": [],
        "scaffold_level": 3,
        "tutorial_visits": 0,
        "last_visit": None,
        "diagnostic_score": None,
        "history": []
    }

def save_student_model(model):
    """持久化学生模型"""
    with open(STUDENT_MODEL_PATH, "w", encoding="utf-8") as f:
        json.dump(model, f, ensure_ascii=False, indent=2)

def update_mastery(model, ilo_key, score, blind_spot=None):
    """更新掌握度, score: 0.0-1.0, blind_spot: str"""
    model["mastery"][ilo_key] = max(model["mastery"].get(ilo_key, 0.0), score)
    if blind_spot and blind_spot not in model["blind_spots"]:
        model["blind_spots"].append(blind_spot)
    if score < 0.7:
        model["scaffold_level"] = max(0, model["scaffold_level"] - 1)
    model["tutorial_visits"] += 1
    model["last_visit"] = datetime.now().isoformat()
    model["history"].append({
        "ts": model["last_visit"],
        "ilo": ilo_key,
        "score": score,
        "blind_spot": blind_spot
    })
    save_student_model(model)
    return model

# ===== 初始化并演示 =====
model = load_student_model()
print("初始 student_model.json:")
print(json.dumps(model, ensure_ascii=False, indent=2))

# 模拟一次 tutorial 后更新 (轮1失败, 轮2过)
model = update_mastery(model, "ILO1_canvas", 0.4, blind_spot="推理成本漏画")
model = update_mastery(model, "ILO2_dcf", 0.85)
print("\ntutorial 后 student_model.json:")
print(json.dumps(model, ensure_ascii=False, indent=2))


## Hattie 四级 Formative Feedback (Hattie 2007 RER 77(1):81-112)

每次 tutorial 结束后, tutor 按四级反馈 (避免 Self 级表扬, 聚焦 Task/Process/Self-Reg/Feed-Forward):

> Hattie (2007) 元分析: formative feedback 效应量 d=0.79-0.9, 但 Self 级(个人表扬)效应量低甚至负面. 本 tutorial 禁 Self 级表扬.

### [TASK] 任务级反馈 - 关于本次任务的具体错误
- 例: "你的 NPV 符号错误, cashflows[0] 应为 -100 而非 100. IRR 计算前提是至少一次符号变化."
- 例: "你的画布漏了推理成本. AI SaaS 毛利率因推理成本被拉低到 65%, 而 HubSpot 是 78%."
- 例: "你的天道推演只做了一层(immediate), 没推演 near/far 三层, 导致 Bull 路径缺乏时间深度."

### [PROCESS] 过程级反馈 - 关于解题策略/方法选择
- 例: "你用点估计 DCF 得到单一 NPV, 但 AI 项目参数不确定性高, 应该用 scipy.stats 蒙特卡洛得 P(NPV>0) 分布."
- 例: "你的敏感性分析只改了一个参数, 应该用龙卷风图对营收/毛利率/增长率/推理成本同时扫描, 识别高杠杆因子."
- 例: "你的天道推演与蒙特卡洛混淆了 - 蒙特卡洛评估参数不确定性(频率派), 天道推演评估场景路径(因果推演), 两者互补."

### [SELF-REG] 自我调节反馈 - 关于元认知/自我监控
- 例: "你在轮3时没有自问'NPV均值和P(NPV>0)是否同分布', 而是直接辩护. 下次先问自己: 这两个数是同一对象的什么统计量?"
- 例: "你的 scaffold_level 已降到 1, 说明连续 3 轮 defense 失败. 你需要在提交前先做 self-test: 能否口述'蒙特卡洛vs天道推演的互补关系'?"
- 例: "你跳过了 pre-task 的决策辩护, 直接进入 tutorial. 下次先完成 pre-task 强制 retrieval, 再来 tutorial."

### [FEED-FORWARD] 前馈反馈 - 关于下一步怎么改
- 例: "下次做 DCF 前, 先列现金流符号检查清单: t=0 必为负, t>=1 为正(或反向). 再调 npf.npv()."
- 例: "你的盲点是'推理成本对估值影响'. 推荐复习: schedule.json C4 卡片 + reading.md 推理成本条目 + 重做 practice.md D3 worked 阶段."
- 例: "下一单元(技能5 Agentic 系统工程)会用今天的投资评估做商业论证. 提前复习: NPV/IRR + 天道推演 Bull/Base/Bear 三路径."
- 例: "24h 后用 schedule.json 的间隔重复卡片自检 C1-C4, 巩固今天暴露的盲点."


## 限频与退出 (防依赖 + exit artifact)

### 限频 (Usage Limit - 防 LLM 依赖)
- **每单元每天 1 次 tutorial (1次/天)**, 防止学生把 tutor 当答案机
- 每次 tutorial 最多 4-5 轮 Socratic 追问, 之后强制退出
- 跨单元复用: student_model.json 记录 scaffold_level, 高 scaffold 学生下次直接进入低脚手架轮
- tutorial_visits > 3 次/单元 触发 '换种学法' 建议(去看 reading.md 或 solution.ipynb worked example)

### 退出条件 (Exit Criteria)
Tutorial 结束学生必须产出 exit artifact:

1. **2-3 个盲点 (blind_spots)** - 列出本次 tutorial 暴露的概念盲点
   - 例: "盲点1: 推理成本对 AI 估值的影响没量化"
   - 例: "盲点2: 天道推演 Bull/Base/Bear 三层推演混淆"
   - 例: "盲点3: NPV 符号方向错误"

2. **推荐复习单元/卡片** - 基于盲点推荐
   - 推理成本盲点 -> 复习 schedule.json C4 + reading.md 推理成本条目
   - 天道推演盲点 -> 复习 notes.md § 天道推演×投资评估 + practice.md D3 worked
   - NPV 符号盲点 -> 复习 practice.md D2 worked + numpy-financial 文档

3. **下次自检承诺** - 24h 后用 schedule.json 的间隔重复卡片自检

### 防依赖机制
- tutor 禁直接答案 (见 cell1 persona: 不直接给答案)
- student_model.json 记录 tutorial_visits, >3 次/单元触发 '换种学法' 建议
- exit artifact 必须学生自己写, tutor 只追问不补全
- 每单元每天 1 次 tutorial (1次/天), 防止高频使用形成依赖

> 本限频机制参考 Oxford tutorial 每周 1 次强制口头辩护传统, 用稀缺性强制学生提前准备 (pre-task retrieval) 而非依赖 tutor.
